# 3D U-Net Training for TripoSR Mesh Fusion

## Important Notes
1. **Different angle counts require different models** - A 3-view model cannot process 5 views
2. **Predictions are denormalized** to match original ground truth dimensions
3. **GPU is used automatically** if available

## Dataset Structure
```
/New_data/TripoSR_{energy}/00000{1-5}_gvxr_processed/view_+{angle}/0/mesh.obj
./Bone_V{1-5}.stl
```

In [ ]:
import os, sys, json, random
import numpy as np
from pathlib import Path
from datetime import datetime
from typing import Optional, List, Dict, Tuple

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader

# Check GPU
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {device}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
else:
    print("WARNING: No GPU detected! Training will be slow.")

In [ ]:
from models.unet3d import get_model
from data.mesh_utils import load_mesh, voxels_to_mesh, NormalizationParams
from data.dataset import TripoSRDataset, AngleCombinationGenerator, VALID_ANGLES
from losses import get_loss_function, ChamferDiceLoss, ChamferDistanceLoss, DiceBCELoss
from metrics import compute_all_metrics, MetricsCalculator

print(f"Valid angles: {len(VALID_ANGLES)} total")
print("Imports OK")

## Configuration

**IMPORTANT**: Change `N_VIEWS` to train for different numbers of angles (2, 3, 4, 5, etc.)

Each different `N_VIEWS` requires a separate trained model!

**MEMORY**: If you have <=12GB GPU, use `USE_LIGHT_MODEL = True`

In [ ]:
# ===== PATHS =====
BASE_PATH = "./New_Result"  # Contains TripoSR_{energy} folders
GT_PATH = "."             # Contains Bone_V1.stl ~ Bone_V5.stl

# ===== ENERGY LEVEL =====
ENERGY_LEVEL = "0.06"  # "" for TripoSR, "0.08" for TripoSR_0.08, etc.

# ===== CRITICAL: NUMBER OF VIEWS =====
# Change this to train for different angle counts
# You need separate models for 2, 3, 4, 5 views etc.
N_VIEWS = 3

# ===== ANGLE CONFIGURATION =====
N_COMBINATIONS = 100       # Combinations per model -> Total = 5 x N
MIN_ANGLE_SEPARATION = 45  # Minimum degrees between angles

# ===== MEMORY-SAVING OPTIONS =====
# Set True if you have <=12GB GPU memory (reduces model from ~8GB to ~2GB)
USE_LIGHT_MODEL = True

# ===== MODEL =====
RESOLUTION = 64            # Voxel resolution (64^3). Use 48 for less memory.
MODEL_TYPE = "attention"   # "standard", "attention", "multiscale"
BASE_FEATURES = 16 if USE_LIGHT_MODEL else 32  # 16 for light, 32 for full
DEPTH = 3 if USE_LIGHT_MODEL else 4            # 3 for light, 4 for full

# ===== LOSS =====
# Options: "dice_bce" (recommended), "chamfer_dice", "dice", "chamfer"
LOSS_TYPE = "dice_bce"

# ===== TRAINING =====
EPOCHS = 100
BATCH_SIZE = 2 if USE_LIGHT_MODEL else 4  # Smaller batch for light model
LEARNING_RATE = 1e-3
TRAIN_SPLIT, VAL_SPLIT = 0.8, 0.1
PATIENCE = 20
SEED = 42

OUTPUT_DIR = "./outputs"

torch.manual_seed(SEED)
np.random.seed(SEED)
random.seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

print("="*60)
print(f"TRAINING CONFIG: {N_VIEWS}-VIEW MODEL")
print("="*60)
print(f"Energy: {ENERGY_LEVEL or 'default'}")
print(f"Views: {N_VIEWS} (model input channels = {N_VIEWS})")
print(f"Combinations: {N_COMBINATIONS} per model")
print(f"Total samples: 5 x {N_COMBINATIONS} = {5 * N_COMBINATIONS}")
print(f"Min angle separation: {MIN_ANGLE_SEPARATION} deg")
print(f"Resolution: {RESOLUTION}^3")
print(f"Loss: {LOSS_TYPE}")
print(f"Device: {device}")
print("="*60)

## Create Dataset

In [ ]:
# Preview angle combinations
gen = AngleCombinationGenerator(N_VIEWS, MIN_ANGLE_SEPARATION)
sample_combos = gen.generate_multiple(5, seed=SEED)
print(f"Sample {N_VIEWS}-view combinations (min {MIN_ANGLE_SEPARATION} deg separation):")
for c in sample_combos:
    print(f"  {c}")

In [ ]:
# Create dataset
dataset = TripoSRDataset(
    base_path=BASE_PATH,
    gt_path=GT_PATH,
    energy_level=ENERGY_LEVEL,
    n_views=N_VIEWS,
    n_combinations=N_COMBINATIONS,
    min_angle_separation=MIN_ANGLE_SEPARATION,
    resolution=RESOLUTION,
    seed=SEED
)

In [ ]:
# Split dataset
n = len(dataset)
n_train = int(n * TRAIN_SPLIT)
n_val = int(n * VAL_SPLIT)
n_test = n - n_train - n_val

train_ds, val_ds, test_ds = torch.utils.data.random_split(
    dataset, [n_train, n_val, n_test],
    generator=torch.Generator().manual_seed(SEED)
)

print(f"Split: Train={n_train}, Val={n_val}, Test={n_test}")

In [ ]:
# Collate function (defined at module level for Windows compatibility)
def collate_fn(batch):
    inputs = torch.stack([b[0] for b in batch])
    targets = torch.stack([b[1] for b in batch])
    metadata = [b[2] for b in batch]
    return inputs, targets, metadata

# Create dataloaders (num_workers=0 for Windows)
train_loader = DataLoader(train_ds, BATCH_SIZE, shuffle=True,
                          num_workers=0, collate_fn=collate_fn, pin_memory=True)
val_loader = DataLoader(val_ds, BATCH_SIZE, shuffle=False,
                        num_workers=0, collate_fn=collate_fn, pin_memory=True)
test_loader = DataLoader(test_ds, BATCH_SIZE, shuffle=False,
                         num_workers=0, collate_fn=collate_fn, pin_memory=True)

print(f"Batches: Train={len(train_loader)}, Val={len(val_loader)}, Test={len(test_loader)}")

In [ ]:
# Test loading
inputs, targets, metadata = next(iter(train_loader))
print(f"Input shape: {inputs.shape} (batch, n_views={N_VIEWS}, D, H, W)")
print(f"Target shape: {targets.shape}")
print(f"\nSample metadata:")
m = metadata[0]
print(f"Model: {m['model_num']}")
print(f"Angles: {m['angles']}")
print(f"GT bounds: {m.get('gt_bounds', 'N/A')}")

## Create Model

In [ ]:
model = get_model(
    model_type=MODEL_TYPE,
    in_channels=N_VIEWS,  # MUST match N_VIEWS!
    out_channels=1,
    base_features=BASE_FEATURES,
    depth=DEPTH
).to(device)

n_params = sum(p.numel() for p in model.parameters())
print(f"Model: {MODEL_TYPE} U-Net")
print(f"Input channels: {N_VIEWS} (for {N_VIEWS}-view reconstruction)")
print(f"Parameters: {n_params:,}")
print(f"Model on GPU: {next(model.parameters()).is_cuda}")

# Test forward
with torch.no_grad():
    out = model(inputs[:1].to(device))
print(f"Forward pass: {inputs[:1].shape} -> {out.shape}")

## Loss and Optimizer

In [ ]:
# Loss function
if LOSS_TYPE == 'chamfer_dice':
    criterion = ChamferDiceLoss(chamfer_weight=0.3, dice_weight=0.7)
elif LOSS_TYPE == 'chamfer':
    criterion = ChamferDistanceLoss()
elif LOSS_TYPE == 'dice_bce':
    criterion = DiceBCELoss(dice_weight=0.5, bce_weight=0.5)
else:
    criterion = get_loss_function(LOSS_TYPE)

# Optimizer with weight decay
optimizer = optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=1e-4)

# Learning rate scheduler
scheduler = optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode='min', factor=0.5, patience=5, verbose=True
)

print(f"Loss: {LOSS_TYPE}")
print(f"Optimizer: AdamW (lr={LEARNING_RATE})")
print(f"Scheduler: ReduceLROnPlateau")

## Training Functions

In [ ]:
def train_epoch(model, loader, criterion, optimizer, device):
    model.train()
    calc = MetricsCalculator()
    total_loss = 0
    
    for batch_idx, (inputs, targets, _) in enumerate(loader):
        inputs = inputs.to(device)
        targets = targets.to(device)
        
        optimizer.zero_grad()
        outputs = model(inputs)
        loss = criterion(outputs, targets)
        loss.backward()
        
        # Gradient clipping
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        
        total_loss += loss.item()
        calc.update(outputs.detach(), targets)
        
        # Progress
        if (batch_idx + 1) % 10 == 0:
            print(f"  Batch {batch_idx+1}/{len(loader)}, Loss: {loss.item():.4f}", end='\r')
    
    m = calc.compute()
    m['loss'] = total_loss / len(loader)
    return m

def eval_epoch(model, loader, criterion, device):
    model.eval()
    calc = MetricsCalculator()
    total_loss = 0
    
    with torch.no_grad():
        for inputs, targets, _ in loader:
            inputs = inputs.to(device)
            targets = targets.to(device)
            outputs = model(inputs)
            loss = criterion(outputs, targets)
            total_loss += loss.item()
            calc.update(outputs, targets)
    
    m = calc.compute()
    m['loss'] = total_loss / len(loader)
    return m

def print_metrics(name, m):
    print(f"{name}: Loss={m['loss']:.4f} | Dice={m['dice']:.3f} | IoU={m['iou']:.3f} | "
          f"PSNR={m['psnr']:.1f}dB | SSIM={m['ssim']:.3f} | CD={m['chamfer_distance']:.4f}")

## Training Loop

In [ ]:
# Output directory
energy_str = ENERGY_LEVEL if ENERGY_LEVEL else "default"
run_name = f"{datetime.now():%Y%m%d_%H%M}_v{N_VIEWS}_E{energy_str}_n{N_COMBINATIONS}"
run_dir = Path(OUTPUT_DIR) / run_name
run_dir.mkdir(parents=True, exist_ok=True)

# Save config
config = {
    'n_views': N_VIEWS,
    'energy_level': ENERGY_LEVEL,
    'n_combinations': N_COMBINATIONS,
    'min_angle_separation': MIN_ANGLE_SEPARATION,
    'resolution': RESOLUTION,
    'model_type': MODEL_TYPE,
    'loss_type': LOSS_TYPE,
    'batch_size': BATCH_SIZE,
    'learning_rate': LEARNING_RATE,
    'total_samples': len(dataset)
}
json.dump(config, open(run_dir / 'config.json', 'w'), indent=2)

print(f"Output: {run_dir}")
print(f"\n{'='*70}")
print(f"TRAINING {N_VIEWS}-VIEW MODEL")
print(f"{'='*70}\n")

In [ ]:
best_loss = float('inf')
patience_cnt = 0
history = {'train': [], 'val': []}

for epoch in range(EPOCHS):
    print(f"\nEpoch {epoch+1}/{EPOCHS}")
    
    # Train
    train_m = train_epoch(model, train_loader, criterion, optimizer, device)
    print()  # Clear progress line
    print_metrics("Train", train_m)
    
    # Validate
    val_m = eval_epoch(model, val_loader, criterion, device)
    print_metrics("Val  ", val_m)
    
    # Scheduler step
    scheduler.step(val_m['loss'])
    
    # Save history
    history['train'].append(train_m)
    history['val'].append(val_m)
    
    # Checkpointing
    if val_m['loss'] < best_loss:
        best_loss = val_m['loss']
        patience_cnt = 0
        torch.save({
            'epoch': epoch,
            'model': model.state_dict(),
            'optimizer': optimizer.state_dict(),
            'config': config,
            'val_metrics': val_m
        }, run_dir / 'best.pt')
        print(f"  -> Saved best model (loss={best_loss:.4f})")
    else:
        patience_cnt += 1
        print(f"  -> No improvement ({patience_cnt}/{PATIENCE})")
        if patience_cnt >= PATIENCE:
            print(f"\nEarly stopping at epoch {epoch+1}")
            break

print(f"\n{'='*70}")
print("Training complete!")
print(f"Best validation loss: {best_loss:.4f}")
print(f"{'='*70}")

## Test Evaluation

In [ ]:
# Load best model
checkpoint = torch.load(run_dir / 'best.pt')
model.load_state_dict(checkpoint['model'])
print(f"Loaded best model from epoch {checkpoint['epoch']+1}")

# Evaluate on test set
test_m = eval_epoch(model, test_loader, criterion, device)

print("\n" + "="*60)
print(f"TEST RESULTS ({N_VIEWS}-VIEW MODEL)")
print("="*60)
print(f"Dice:             {test_m['dice']:.4f} (higher=better)")
print(f"IoU:              {test_m['iou']:.4f} (higher=better)")
print(f"PSNR:             {test_m['psnr']:.2f} dB (higher=better)")
print(f"SSIM:             {test_m['ssim']:.4f} (higher=better)")
print(f"Chamfer Distance: {test_m['chamfer_distance']:.6f} (lower=better)")
print(f"RMSE:             {test_m['rmse']:.6f} (lower=better)")
print("="*60)

# Save results
json.dump(test_m, open(run_dir / 'test_results.json', 'w'), indent=2)

## Export Predictions with Proper Denormalization

This exports predictions back to the original coordinate system of the ground truth.

In [ ]:
def denormalize_mesh_to_gt(pred_mesh, gt_center, gt_scale):
    """
    Transform predicted mesh from normalized space [-0.5, 0.5]^3
    back to original ground truth coordinate system.
    """
    mesh = pred_mesh.copy()
    
    # Reverse normalization: 
    # Original: vertices = (vertices - center) * scale
    # Inverse: vertices = vertices / scale + center
    if gt_scale != 0:
        mesh.vertices = mesh.vertices / gt_scale
    mesh.vertices = mesh.vertices + np.array(gt_center)
    
    return mesh

def export_prediction(model, sample_input, metadata, output_path, device, threshold=0.5):
    """
    Export prediction as mesh in original GT coordinate system.
    """
    model.eval()
    with torch.no_grad():
        if sample_input.dim() == 4:
            sample_input = sample_input.unsqueeze(0)
        pred = model(sample_input.to(device))[0, 0].cpu().numpy()
    
    # Convert to mesh (in normalized space)
    pred_mesh = voxels_to_mesh(pred, threshold=threshold)
    
    # Denormalize to GT coordinate system
    if 'gt_center' in metadata and 'gt_scale' in metadata:
        pred_mesh = denormalize_mesh_to_gt(
            pred_mesh, 
            metadata['gt_center'], 
            metadata['gt_scale']
        )
        print(f"  Denormalized to GT space")
        print(f"    Center: {metadata['gt_center']}")
        print(f"    Bounds: {metadata.get('gt_bounds', 'N/A')}")
    
    # Export
    pred_mesh.export(str(output_path))
    print(f"  Saved: {output_path}")
    print(f"  Vertices: {len(pred_mesh.vertices)}, Faces: {len(pred_mesh.faces)}")
    
    return pred_mesh

In [ ]:
# Export one prediction per model
export_dir = run_dir / 'predictions'
export_dir.mkdir(exist_ok=True)

exported_models = set()

for inputs, targets, metadata in test_loader:
    for i, m in enumerate(metadata):
        model_num = m['model_num']
        if model_num in exported_models:
            continue
        exported_models.add(model_num)
        
        print(f"\nExporting Model {model_num}:")
        print(f"  Angles: {m['angles']}")
        
        out_path = export_dir / f'pred_model{model_num}_v{N_VIEWS}.stl'
        export_prediction(model, inputs[i], m, out_path, device)
        
        if len(exported_models) >= 5:
            break
    if len(exported_models) >= 5:
        break

print(f"\nExported {len(exported_models)} predictions to: {export_dir}")

## Visualization

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(2, 3, figsize=(15, 8))
metrics_plot = ['loss', 'dice', 'iou', 'psnr', 'ssim', 'chamfer_distance']

for ax, metric in zip(axes.flat, metrics_plot):
    ax.plot([h[metric] for h in history['train']], label='Train', linewidth=2)
    ax.plot([h[metric] for h in history['val']], label='Val', linewidth=2)
    ax.set_title(metric.upper(), fontsize=12)
    ax.legend()
    ax.grid(True, alpha=0.3)

plt.suptitle(f'{N_VIEWS}-View Model Training History', fontsize=14)
plt.tight_layout()
plt.savefig(run_dir / 'training_history.png', dpi=150)
plt.show()

In [ ]:
# Visualize predictions vs GT
model.eval()
inputs, targets, metadata = next(iter(test_loader))

with torch.no_grad():
    preds = model(inputs.to(device)).cpu()

fig, axes = plt.subplots(min(4, BATCH_SIZE), 4, figsize=(16, 4*min(4, BATCH_SIZE)))
if min(4, BATCH_SIZE) == 1:
    axes = axes.reshape(1, -1)

mid = RESOLUTION // 2

for i in range(min(4, BATCH_SIZE)):
    pred = preds[i, 0].numpy()
    tgt = targets[i, 0].numpy()
    diff = np.abs(pred - tgt)
    
    axes[i, 0].imshow(pred[mid], cmap='gray')
    axes[i, 0].set_title(f'Prediction (Model {metadata[i]["model_num"]})')
    axes[i, 0].axis('off')
    
    axes[i, 1].imshow(tgt[mid], cmap='gray')
    axes[i, 1].set_title('Ground Truth')
    axes[i, 1].axis('off')
    
    axes[i, 2].imshow(diff[mid], cmap='hot')
    axes[i, 2].set_title('Difference')
    axes[i, 2].axis('off')
    
    axes[i, 3].imshow((pred[mid] > 0.5).astype(float), cmap='gray')
    axes[i, 3].set_title(f'Binary (angles: {metadata[i]["angles"]})')
    axes[i, 3].axis('off')

plt.suptitle(f'{N_VIEWS}-View Predictions (Axial Slice)', fontsize=14)
plt.tight_layout()
plt.savefig(run_dir / 'predictions_visual.png', dpi=150)
plt.show()

## Summary

In [ ]:
print("\n" + "="*70)
print(f"SUMMARY: {N_VIEWS}-VIEW MODEL")
print("="*70)
print(f"\nConfiguration:")
print(f"  Views: {N_VIEWS}")
print(f"  Energy: {ENERGY_LEVEL or 'default'}")
print(f"  Samples: {len(dataset)} (5 models x {N_COMBINATIONS} combinations)")
print(f"  Resolution: {RESOLUTION}^3")
print(f"  Model: {MODEL_TYPE}")
print(f"  Loss: {LOSS_TYPE}")
print(f"\nTest Results:")
print(f"  Dice: {test_m['dice']:.4f}")
print(f"  IoU: {test_m['iou']:.4f}")
print(f"  PSNR: {test_m['psnr']:.2f} dB")
print(f"  SSIM: {test_m['ssim']:.4f}")
print(f"  Chamfer: {test_m['chamfer_distance']:.6f}")
print(f"\nOutput: {run_dir}")
print(f"\nTo use this model for inference with {N_VIEWS} views:")
print(f"  checkpoint = torch.load('{run_dir}/best.pt')")
print(f"  model.load_state_dict(checkpoint['model'])")
print("="*70)

---

## **Training Multiple View Models

To train models for different numbers of views, change `N_VIEWS` and re-run:

```python
# For 2-view model
N_VIEWS = 2
MIN_ANGLE_SEPARATION = 90  # Larger separation for fewer views

# For 4-view model  
N_VIEWS = 4
MIN_ANGLE_SEPARATION = 45

# For 5-view model
N_VIEWS = 5
MIN_ANGLE_SEPARATION = 36

# For 6-view model
N_VIEWS = 6
MIN_ANGLE_SEPARATION = 30
```

Each model will be saved with its view count in the filename.